#Lab.14 / IBM3202 – Molecular Docking of Multiple Ligands (Virtual Screening)

###Theoretical aspects

As commented on [Lab.06](https://colab.research.google.com/github/pb3lab/ibm3202/blob/master/tutorials/lab06_docking.ipynb), **molecular docking** explores the potential binding poses of small molecules on the **binding site** of a target protein for which an experimentally determined structure is available.

When used for drug discovery, typically one would like to perform the docking of many molecules against a given target protein to identify those that are most likely to bind to it.

This strategy is called **virtual screening** and will be explored in this tutorial.

This tutorial is based on an article published in _Protein Science_ in 2023 ([Rivera et al, _Protein Sci_. 2023, 32(7):e4706](https://doi.org/10.1002/pro.4706)), in which molecular docking was used to evaluate the binding free energy between a modeled protein structure of the BiP chaperone with magnesium in its binding site and different nucleotides (ATP, ADP, etc).

#Part 0 – Downloading and Installing the required software

Before we start, you must first **remember to start the hosted runtime in Google Colab**.

Then, we must install several pieces of software to perform our task. Namely:
- **py3Dmol** for visualization of the protein structure and setting up the search grid.
- **biopython** for downloading and manipulating protein structures
- **miniconda**, a free minimal installer of **conda** for software package and environment management.
- **ChEMBL web resource client** for downloading the ligands
- **OpenBabel** for manipulation of our ligand(s).
- **meeko** for parameterization of our target protein and ligands for docking.
- **Autodock Vina** for the docking process
- **QuickVina** for faster docking processes

  After several tests, the following installation instructions are the best way of setting up **Google Colab** for this laboratory session.

1. We will first install py3Dmol, biopython, ChEMBL and meeko as follows:

In [ ]:
#Installing py3Dmol using pip
!pip -q install py3Dmol
#Installing biopython using pip
!pip -q install biopython
#We will also install kora for using RDkit
!pip -q install kora rdkit
#Finally, we will install ChEMBL
!pip -q install chembl_webresource_client
#Installing meeko
!pip -q install meeko

In [ ]:
#Importing py3Dmol for safety
import py3Dmol

2. And then we will install conda to be able to install OpenBabel

In [ ]:
#Install conda using the new conda-colab library
!pip install -q condacolab
import condacolab
condacolab.install()

#Install OpenBabel from the bioconda repository
!mamba install --quiet --log-level error -c conda-forge -c bioconda openbabel scipy gemmi zlib ncurses --yes

3. We will download the Autodock Vina program from the [AutoDock Vina GitHub](https://github.com/ccsb-scripps/AutoDock-Vina) and make an alias to use it during this session

In [ ]:
#Download and extract Autodock Vina from SCRIPPS
#Then, we set up an alias for vina to be treated as a native binary
%%bash
mkdir vina
cd vina
wget -q https://github.com/ccsb-scripps/AutoDock-Vina/releases/download/v1.2.7/vina_1.2.7_linux_x86_64
chmod +x /content/vina/vina_1.2.7_linux_x86_64
wget -q https://github.com/ccsb-scripps/AutoDock-Vina/releases/download/v1.2.7/vina_split_1.2.7_linux_x86_64
chmod +x /content/vina/vina_split_1.2.7_linux_x86_64

In [ ]:
%alias vina /content/vina/vina_1.2.7_linux_x86_64
%alias vina_split /content/vina/vina_1.2.7_linux_x86_64

4. Finally, we will download QuickVina 2 from the [QuickVina GitHub](https://github.com/QVina/qvina), a redesigned version of AutoDock Vina that is 20 times faster and equally accurate.


In [ ]:
%%bash
#Downloading Quick Vina and enabling its execution
git clone https://github.com/QVina/qvina
chmod +x /content/qvina/bin/qvina2.1

In [ ]:
%alias qvina2 /content/qvina/bin/qvina2.1

Once these software installation processes are completed, we are ready to perform our experiments

#Part 1 – Preparing the Receptor for Virtual Screening with rigid or flexible side chains

1. The first step in a molecular docking procedure is to have a structure of a given target protein. While in some cases a high-quality comparative model will be used, most cases start with an experimentally (X-ray, NMR, cryoEM) solved three-dimensional structure.

    For this tutorial, we will download a high-quality template-modelling generated structure of the BiP chaperone bound to a magnesium ion. First, let's make a folder for our virtual screening, and then let's download the structure into this folder

In [ ]:
#Let's make sure we are on the main directory
import os
os.chdir('/content/')

#Let's make a folder first. We need to import the os and path library
from pathlib import Path

#Then, we define the path of the folder we want to create.
#Notice that the HOME folder for a hosted runtime in colab is /content/
singlepath = Path("/content/VSdocking/")

#Now, we create the folder using the os.mkdir() command
#The if conditional is just to check whether the folder already exists
#In which case, python returns an error
if os.path.exists(singlepath):
  print("Virtual screening path already exists")
if not os.path.exists(singlepath):
  os.mkdir(singlepath)
  print("Virtual screening path was succesfully created")

#Now we will change to the new folder
os.chdir(singlepath)

#Download PDB file from GitHub
!wget -q https://raw.githubusercontent.com/pb3lab/ibm3202/refs/heads/master/input_files/robetta_Trmodels_52159_3_plusMG.pdb -O BiP.pdb
print("PDB file successfully downloaded")

2. For AutoDock to perform a molecular docking experiment, the protein target must contain information about the partial charges of each atom and atom types that are compatible with AutoDock. Such format is referred to as **PDBQT**, a modification of the PDB format that also includes **charges (q)** and **AutoDock-specific atom types (t)** in two extra columns at the end of the now PDBQT file.

    Lastly, the protein target must contain **all polar hydrogens**. Most protein structures have no hydrogens included, meaning that we must add them. We will add polar hydrogens of the protein and parameterize it based on the pKa of each aminoacid at pH 7.4 with the **AMBER99ff** force field using **pdb2pqr**, followed by deletion of non-polar hydrogens and conversion into **PDBQT** file using **MGLtools**. This is mostly because MGLTools does not have parameters for Magnesium to generate the PDQBT file.

In [ ]:
# Titrate protein sidechains and generate PDBQT directly, preserving all ions/metals natively
!mk_prepare_receptor.py --compute_charges --read_pdb $singlepath/BiP.pdb -p $singlepath/BiP.pdbqt

#Part 2 – Downloading and Preparing the Ligands for AutoDock

1. We will first start by creating a different folder for each ligand, in which we will store our ligands separately for molecular docking.

In [ ]:
#Let's make a folder first. We need to import the os and path library
import os
from pathlib import Path

#We will first create a path for all ligands that we will use in this tutorial
#Notice that the HOME folder for a hosted runtime in colab is /content/
#Ligands = ATP, AMP-PNP, ATPgammaS, ADP

ligands = ['CHEMBL14249', 'CHEMBL2220361', 'CHEMBL131890', 'CHEMBL14830']

for l in ligands:
  ligandpath = Path("/content/" + l)
  if os.path.exists(ligandpath):
    print("ligand path already exists")
  if not os.path.exists(ligandpath):
    os.mkdir(ligandpath)
  print("ligand path " + str(ligandpath) + " was succesfully created")

2. Now, we will download ATP (CHEMBL14249), AMP-PNP (CHEMBL2220361), ATP$\gamma$S (CHEMBL131890) and ADP (CHEMBL14830) from the **ChEMBL** database (https://www.ebi.ac.uk/chembl/). This is a comprehensive, freely accessible, online database containing information on different compounds, drugs and drug targets.

  We will download this ligand in SMILES format to continue with its preparation for molecular docking

In [ ]:
#Downloading Ligands from ChEMBL database
from chembl_webresource_client.new_client import new_client
molecule = new_client.molecule
ligands = ['CHEMBL14249', 'CHEMBL2220361', 'CHEMBL131890', 'CHEMBL14830']
for l in ligands:
  ligandpath = Path("/content/" + l)
  with open(ligandpath / "ligand.smiles","w") as f:
    m1 = molecule.get(l)
    f.write(m1['molecule_structures']['canonical_smiles'])

3. **Let's take a look at the SMILES of each molecule**

In [ ]:
#Print the SMILES of all ligands
for l in ligands:
  ligandpath = Path("/content/" + l)
  print(l)
  print((ligandpath / "ligand.smiles").read_text())

In [ ]:
#@title Use the following Viewer to load your SMILES as a 3D molecule
import py3Dmol
import rdkit
from rdkit import Chem
from rdkit.Chem import AllChem

def MolTo3DView(mol, size=(300, 300), style="stick", surface=False, opacity=0.5):
    assert style in ('line', 'stick', 'sphere', 'carton')
    mblock = Chem.MolToMolBlock(mol)
    viewer = py3Dmol.view()
    viewer.addModel(mblock, 'mol')
    viewer.setStyle({style:{}})
    if surface:
        viewer.addSurface(py3Dmol.SAS, {'opacity': opacity})
    viewer.zoomTo()
    return viewer

from ipywidgets import interact,fixed,IntSlider
import ipywidgets

def smi2conf(smiles):
    '''Convert SMILES to rdkit.Mol with 3D coordinates'''
    mol = Chem.MolFromSmiles(smiles)
    if mol is not None:
        mol = Chem.AddHs(mol)
        AllChem.EmbedMolecule(mol)
        AllChem.MMFFOptimizeMolecule(mol, maxIters=200)
        return mol
    else:
        return None

@interact
def smi2viewer(smi='CC=O'):
    try:
        conf = smi2conf(smi)
        return MolTo3DView(conf).show()
    except:
        return None

4. Now, we will take this SMILES format and use it to construct and parameterize a three-dimensional structure of all ligands in **PDBQT** format for its use in molecular docking. As with the receptor, we will use **meeko** again for this process.

In [ ]:
import os
from pathlib import Path

for l in ligands:
    ligandpath = Path("/content") / l
    os.chdir(ligandpath)

    # 1. Generate 3D coordinates and protonate at pH 7.4 with OpenBabel -> output SDF
    !obabel ligand.smiles -O ligand.sdf --gen3d best -p 7.4 --canonical

    # 2. Parameterize with Meeko: detects rotatable bonds, merges non-polar H,
    # assigns Gasteiger charges, and writes PDBQT
    print("Preparing ligand:", ligandpath)
    !mk_prepare_ligand.py -i ligand.sdf -o ligand.pdbqt

    # 3. Clean up the temporary intermediate SDF
    if Path("ligand.sdf").exists():
        os.remove("ligand.sdf")

os.chdir("/content")

**You are all set with your ligand!** Now, we move onto setting up the molecular docking experiment

#Part 3 – Setting up and Performing Virtual Screening with AutoDock

1. As explained in the lectures, it is necessary to define the search space for molecular docking on a given target protein through the use of a **grid box**. This grid box is usually centered within the binding, active or allosteric site of the target protein and its size will be sufficiently large such that **all binding residues are placed inside the grid box**.

    Here, we will make use of **py3Dmol** to visually inspect the protein structure in cartoon representation and to draw a grid box. The position and size of the grid box will be defined by the coordinates of its centroid and by its dimensions in x, y and z.

    To better guide the search for the optimal dimensions and coordinates of the grid box, we will also show the residues 38, 72, 176, 234, 268, 271, 272 and 275 of the BiP chaperone.

    The script that defines the visualizer, which we called **ViewProtGrid**, is first loaded into **Colab** with the following lines of code

In [ ]:
#@title Loading the script that creates our Docking Box Viewer
#These definitions will enable loading our protein and then
#drawing a box with a given size and centroid on the cartesian space
#This box will enable us to set up the system coordinates for the simulation
#
#HINT: The active site of the HIV-2 protease is near the beta strands in green
#
#ACKNOWLEDGE: This script is largely based on the one created by Jose Manuel
#Napoles Duarte, Physics Teacher at the Chemical Sciences Faculty of the
#Autonomous University of Chihuahua (https://github.com/napoles-uach)
#
#First, we define the grid box
def definegrid(object,bxi,byi,bzi,bxf,byf,bzf):
  object.addBox({'center':{'x':bxi,'y':byi,'z':bzi},'dimensions': {'w':bxf,'h':byf,'d':bzf},'color':'blue','opacity': 0.6})

#Next, we define how the protein will be shown in py3Dmol
#Note that we are also adding a style representation for active site residues
def viewprot(object,prot_PDBfile,resids):
  mol1 = open(prot_PDBfile, 'r').read()
  object.addModel(mol1,'pdb')
  object.setStyle({'cartoon': {'color':'spectrum'}})
  object.addStyle({'resi':resids},{'stick':{'colorscheme':'greenCarbon'}})

#Lastly, we combine the box grid and protein into a single viewer
def viewprotgrid(prot_PDBfile,resids,bxi,byi,bzi,bxf=10,byf=10,bzf=10):
  mol_view = py3Dmol.view(1000,1000,viewergrid=(1,2))
  definegrid(mol_view,bxi,byi,bzi,bxf,byf,bzf)
  viewprot(mol_view,prot_PDBfile,resids)
  mol_view.setBackgroundColor('0xffffff')
  mol_view.rotate(90, {'x':0,'y':1,'z':0},viewer=(0,1));
  mol_view.zoomTo()
  mol_view.show()

2. Now, we will use our ViewProtGrid to visualize the protein, binding site residues and a grid box of variable size and position that we can manipulate using a slider through *ipywidgets*. You have to enter the location of the PDB file in the *pdbfile* variable (e.g. single-dock/1hsg_prot.pdb) and the residues that you want to show from the PDB in the *resids* variable.


Examples of how to use the *pdbfile* variable
>pdbfile = 1hsg_prot.pdb (if the PDB file is in the current path)

>pdbfile = single-dock/1hsg_prot.pdb (if the PDB file is in the single-dock folder)

Examples of how to use the *resids* variable

>resids = [82] shows a single residue in position 82)

>resids = [82,83,84] shows residues 82, 83 or 84 separately, which you can select in the viewer

>resids = [(82,83,84)] shows residue 82, 83 and 84 in the same visualization

>resids = ['82-84'] shows residue range 82-84 in the same visualization

**NOTE:** This code fails when attempting to show two non-consecutive residues in the same visualization.

In [ ]:
#@title Loading the Docking Box viewer for a user-defined PDB input file and highlighted residues
from ipywidgets import interact,fixed,IntSlider
import ipywidgets

pdbfile = "VSdocking/BiP.pdb" # @param {type:"string"}
resids = [(38,72,176,234,268,271,272,275)] # @param {type:"raw"}
interact(viewprotgrid,
#ADD YOUR PDB LOCATION AND FILENAME HERE
         prot_PDBfile = pdbfile,
#ADD THE RESIDUES YOU WANT TO VISUALIZE HERE
         resids = resids,
         bxi=ipywidgets.IntSlider(min=-100,max=100, step=1),
         byi=ipywidgets.IntSlider(min=-100,max=100, step=1),
         bzi=ipywidgets.IntSlider(min=-100,max=100, step=1),
         bxf=ipywidgets.IntSlider(min=4,max=30, step=1),
         byf=ipywidgets.IntSlider(min=4,max=30, step=1),
         bzf=ipywidgets.IntSlider(min=4,max=30, step=1))

3. Now, we will generate a configuration file for **Autodock**. As expected, the configuration file contains information about the target protein and ligand, as well as the position and dimensions of the grid box that defines the search space.

    After careful inspection of an adequate box grid, the origin of the box for our comparative model [-17, 7, -33] and its size is [24, 24, 24].

    For defining the grid box, you will use the box origin and size coordinates that you defined manually in the previous step.

    The following is an example file of a standard **Autodock configuration file**, including all possible variables that can be edited:


```
#CONFIGURATION FILE

#INPUT OPTIONS
receptor = [target protein pdbqt file]
ligand = [ligand pdbqt file]
flex = [flexible residues in receptor in pdbqt format]

#SEARCH SPACE CONFIGURATIONS
#Center of the box (coordinates x, y and z
center_x = [value]
center_y = [value]
center_z = [value]
#Size of the box (dimensions in x, y and z)
size_x = [value]
size_y = [value]
size_z = [value]

#OUTPUT OPTIONS
#out = [output pdbqt file for all conformations]
#log = [output log file for binding energies]

#OTHER OPTIONS
cpu = [value] # more cpus reduces the computation time
exhaustiveness = [value] # search time for finding the global minimum, default is 8
num_modes = [value] # maximum number of binding modes to generate, default is 9
energy_range = [value] # maximum energy difference between the best binding mode and the worst one displayed (kcal/mol), default is 3
seed = [value] # explicit random seed, not required
```

The following script will create this file for our docking procedure. **You will need to add the position and dimensions of your grid box**


In [ ]:
#@title Generating an AutoDock Vina file
receptor = "/content/VSdocking/BiP.pdbqt" # @param {type:"string"}
ligand = "ligand.pdbqt" # @param {type:"string"}
center_x = "-17" # @param {type:"string"}
center_y = "7" # @param {type:"string"}
center_z = "-33" # @param {type:"string"}
size_x = "24" # @param {type:"string"}
size_y = "24" # @param {type:"string"}
size_z = "24" # @param {type:"string"}
with open(singlepath / "config_singledock","w") as f:
  f.write("#CONFIGURATION FILE (options not used are commented) \n")
  f.write("\n")
  f.write("#INPUT OPTIONS \n")
  f.write("receptor = " + receptor +"\n")
  f.write("ligand = " + ligand +"\n")
  f.write("#flex = [flexible residues in receptor in pdbqt format] \n")
  f.write("#SEARCH SPACE CONFIGURATIONS \n")
  f.write("#Center of the box (values bxi, byi and bzi) \n")
#CHANGE THE FOLLOWING DATA WITH YOUR BOX CENTER COORDINATES
  f.write("center_x = " + center_x + "\n")
  f.write("center_y = " + center_y + "\n")
  f.write("center_z = " + center_z + "\n")
#CHANGE THE FOLLOWING DATA WITH YOUR BOX DIMENSIONS
  f.write("#Size of the box (values bxf, byf and bzf) \n")
  f.write("size_x = " + size_x + "\n")
  f.write("size_y = " + size_y + "\n")
  f.write("size_z = " + size_z + "\n")
  f.write("#OUTPUT OPTIONS \n")
  f.write("#out = \n")
  f.write("#log = \n")
  f.write("\n")
  f.write("#OTHER OPTIONS \n")
  f.write("#cpu =  \n")
  f.write("#exhaustiveness = \n")
  f.write("#num_modes = \n")
  f.write("#energy_range = \n")
  f.write("#seed = ")

4. Lastly, we will enter into the folder that we created for the docking experiment and **perform our first molecular docking with Autodock**.

    Once you execute the lines of code shown below, Autodock will show you a progress bar (if running as expected).
  
    Note that we are defining the filenames of the output and log file outside the configuration file.

In [ ]:
#Changing directory to each ligand folder for docking
for l in ligands:
  ligandpath = Path("/content/" + l)
  os.chdir(ligandpath)
#Executing AutoDock Vina with our configuration file
  %vina --config $singlepath/config_singledock --out output.pdbqt | tee output.log
#Exiting the execution directory
os.chdir("/content/")

5. Want to run faster? Then use QuickVina 2!

In [ ]:
#Changing directory to each ligand folder for docking
for l in ligands:
  ligandpath = Path("/content/" + l)
  os.chdir(ligandpath)
#Executing AutoDock Vina with our configuration file
  %qvina2 --config $singlepath/config_singledock --out output.pdbqt | tee output.log
#Exiting the execution directory
os.chdir("/content/")

6. We will split the different docking poses generated as a result of the molecular docking simulation into separate PDB files using **babel**, starting with file numbered as 1 corresponding to the lowest-energy pose.

In [ ]:
#We need to convert our Autodock Vina results from pdbqt into pdb
#For this, we use babel
#Change to the ligand directory
for l in ligands:
  ligandpath = Path("/content/" + l)
  os.chdir(ligandpath)
#Using babel to split the configurations
  !obabel -ipdbqt output.pdbqt -opdb -O {l}_.pdb -m
#Go back to the home directory
os.chdir("/content/")

7. Finally, we create another visualizer (**ViewDocking**) to load our protein and any docking pose of our choice.

In [ ]:
import py3Dmol

def viewdocking(protein_name, ligand_name):
    mview = py3Dmol.view(width=800, height=400)

    with open(protein_name, 'r') as f:
        mol1 = f.read()
    with open(ligand_name, 'r') as f:
        mol2 = f.read()

    # Determine formats from extension (handles pdb, pdbqt, mol2, sdf)
    fmt1 = protein_name.split('.')[-1].lower()
    fmt2 = ligand_name.split('.')[-1].lower()

    # Model 0: Receptor
    mview.addModel(mol1, fmt1)
    mview.setStyle({'model': 0}, {'cartoon': {'color': 'spectrum'}})
    mview.addStyle({'model': 0, 'resn': 'MG'}, {'sphere': {'color': 'yellow', 'radius': 1.2}})

    # Model 1: Ligand (top pose)
    mview.addModel(mol2, fmt2)
    mview.setStyle({'model': 1}, {'stick': {'colorscheme': 'greenCarbon'}})

    mview.setBackgroundColor('white')
    mview.zoomTo({'model': 1})  # Zooms into the binding pocket / ligand
    return mview.show()

7. The ViewDocking visualizer can then be used as indicated below.

In [ ]:
#View docking results
#viewdocking('protein_file','docked_ligand_file')
viewdocking('VSdocking/BiP.pdb','CHEMBL131890/CHEMBL131890_1.pdb')

It is **recommended** to visualize these docking results in a standalone software (e.g. VMD, PyMOL, Chimera, etc). Thus, we will generate compressed ZIP files for downloading these results. You can then right-click on the generated ZIP files and select *Download* in the drop-down menu.


In [ ]:
# Bundle VSdocking and all CHEMBL folders into one archive
!zip -r /content/all_docking_results.zip /content/VSdocking /content/CHEMBL*
from google.colab import files
files.download("/content/all_docking_results.zip")